## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
import numpy as np
tf.enable_eager_execution()

### Collect Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd

In [0]:
df1 = pd.read_csv("/content/drive/My Drive/prices.csv")

### Check all columns in the dataset

In [6]:
df1.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Drop columns `date` and  `symbol`

In [0]:
df1 = df1.drop(['date', 'symbol'], axis = 1)

In [8]:
df1.head()

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


In [9]:
df1.shape

(851264, 5)

### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [0]:
df1t = df1[:1000]

In [11]:
df1t.shape

(1000, 5)

### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
# split into input (X) and output (Y) variables
X = df1t.iloc[:,0:4]
Y = df1t.iloc[:,[4]]
#split into 67% for train and 33% for test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=7)

In [14]:
# split into input (X) and output (Y) variables
print(X.head())

print(Y.head())

         open       close         low        high
0  123.430000  125.839996  122.309998  126.250000
1  125.239998  119.980003  119.940002  125.540001
2  116.379997  114.949997  114.930000  119.739998
3  115.480003  116.620003  113.500000  117.440002
4  117.010002  114.970001  114.089996  117.330002
      volume
0  2163600.0
1  2386400.0
2  2489500.0
3  2006300.0
4  1408600.0


#### Convert Training and Test Data to numpy float32 arrays


In [0]:
import numpy as np

In [0]:
X_train = np.float32(X_train)
X_test = np.float32(X_test)
y_train = np.float32(y_train)
y_test= np.float32(y_test) 

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
from sklearn.preprocessing import normalize

X_train = normalize(np.vstack([X_train]), norm='l2', axis=1)
X_test = normalize(np.vstack([X_test]), norm='l2', axis=1)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.random_normal(shape=[4,1])
b = tf.zeros(shape=[1])

In [19]:
import tensorflow as tf
W.numpy()

array([[ 1.9677166 ],
       [ 2.1858313 ],
       [ 0.78383625],
       [-0.5629888 ]], dtype=float32)

2.Define a function to calculate prediction

In [0]:
y = tf.add(tf.matmul(X_train,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [21]:
loss = tf.reduce_mean(tf.square(y_train-y),name='Loss')
loss

<tf.Tensor: id=17, shape=(), dtype=float32, numpy=317998570000000.0>

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03)

In [23]:
W


<tf.Tensor: id=5, shape=(4, 1), dtype=float32, numpy=
array([[ 1.9677166 ],
       [ 2.1858313 ],
       [ 0.78383625],
       [-0.5629888 ]], dtype=float32)>

In [24]:
b

<tf.Tensor: id=8, shape=(1,), dtype=float32, numpy=array([0.], dtype=float32)>

In [25]:
train_op

In [0]:
#how many times data need to be shown to model
training_epochs = 100

In [0]:
def loss(X_train,y_train,W,b):
  yPred = tf.add(tf.matmul(X_train,W),b)
  return tf.reduce_mean(tf.square(y_train-yPred))

In [0]:
def training(X,W,b,y,learning_value):
    with tf.GradientTape() as t:
       t.watch([W,b])
       loss1 = loss(X,y,W,b)
    dw,db=t.gradient(loss1,[W,b])
    w=W-learning_value*dw
    b=b-learning_value*db
    return w,b
  

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [30]:
for epoch in range(training_epochs):
            
#     #Calculate train_op and loss
#     _, train_loss = sess.run([train_op,loss],{x:X_train, y_:y_train})
    
#     if epoch % 5 == 0:
#         print ('Training loss at step: ', epoch, ' is ', train_loss)

    Wcalc,bcalc = training(X_train,W,b,y_train,0.3)
  
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', loss(X_train,y_train,Wcalc,bcalc).numpy())

Training loss at step:  0  is  286774900000000.0
Training loss at step:  5  is  286774900000000.0
Training loss at step:  10  is  286774900000000.0
Training loss at step:  15  is  286774900000000.0
Training loss at step:  20  is  286774900000000.0
Training loss at step:  25  is  286774900000000.0
Training loss at step:  30  is  286774900000000.0
Training loss at step:  35  is  286774900000000.0
Training loss at step:  40  is  286774900000000.0
Training loss at step:  45  is  286774900000000.0
Training loss at step:  50  is  286774900000000.0
Training loss at step:  55  is  286774900000000.0
Training loss at step:  60  is  286774900000000.0
Training loss at step:  65  is  286774900000000.0
Training loss at step:  70  is  286774900000000.0
Training loss at step:  75  is  286774900000000.0
Training loss at step:  80  is  286774900000000.0
Training loss at step:  85  is  286774900000000.0
Training loss at step:  90  is  286774900000000.0
Training loss at step:  95  is  286774900000000.0


### Get the shapes and values of W and b

In [31]:
W

<tf.Tensor: id=5, shape=(4, 1), dtype=float32, numpy=
array([[ 1.9677166 ],
       [ 2.1858313 ],
       [ 0.78383625],
       [-0.5629888 ]], dtype=float32)>

In [33]:
W.shape

TensorShape([Dimension(4), Dimension(1)])

In [34]:
b

<tf.Tensor: id=8, shape=(1,), dtype=float32, numpy=array([0.], dtype=float32)>

In [35]:
b.shape

TensorShape([Dimension(1)])

### Model Prediction on 1st Examples in Test Dataset

In [0]:
def predict(x,w,b):
    return tf.add(tf.matmul(x,w),b)
  
y_predict = predict(X_test[0:1], W, b)


In [42]:
y_predict

<tf.Tensor: id=7193, shape=(1, 1), dtype=float32, numpy=array([[2.17402]], dtype=float32)>

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
df = pd.read_csv("/content/drive/My Drive/Iris.csv")

In [262]:
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
df = pd.get_dummies(df['Species'])

In [264]:
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,1,5.1,3.5,1.4,0.2,1,0,0
1,2,4.9,3.0,1.4,0.2,1,0,0
2,3,4.7,3.2,1.3,0.2,1,0,0
3,4,4.6,3.1,1.5,0.2,1,0,0
4,5,5.0,3.6,1.4,0.2,1,0,0


### Splitting the data into feature set and target set

In [0]:
# split into input (X) and output (Y) variables
X = df.iloc[:,1:5]
#Y = df.iloc[:,6:9]
Y = df[["Species_Iris-setosa", "Species_Iris-versicolor", "Species_Iris-virginica"]]
#split into 67% for train and 33% for test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.33, random_state=7)


In [286]:
X.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [289]:
Y.head()

,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0


In [290]:
X_test.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
149,5.9,3.0,5.1,1.8
84,5.4,3.0,4.5,1.5
40,5.0,3.5,1.3,0.3
66,5.6,3.0,4.5,1.5
106,4.9,2.5,4.5,1.7


In [292]:
y_test.head()

,Species_Iris-setosa,Species_Iris-versicolor,Species_Iris-virginica
149,0,0,1
84,0,1,0
40,1,0,0
66,0,1,0
106,0,0,1


###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
import numpy
from pandas import read_csv
from keras.models import Sequential
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.layers import Flatten
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [0]:
# create model
model = tf.keras.Sequential()
#model.add(Dense(2, input_dim=4, activation='relu')) 
model.add(tf.keras.layers.Dense(3, input_shape=(4,),activation='softmax')) # Output layer 
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])
# there are 2 cross Entorpy - Binary_crossentrophu for binary class model for Multiple class- Categorical class entrophy
# Classification use Entorphy and for Regression use mse 
#optimizer  in next class
# Metric = Accuracy 


### Model Training 

In [295]:
model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=200)

Train on 100 samples, validate on 50 samples
Epoch 1/200
100/100 [==============================] - 0s 2ms/sample - loss: 2.0126 - acc: 0.4800 - val_loss: 1.8471 - val_acc: 0.6400
Epoch 2/200
100/100 [==============================] - 0s 235us/sample - loss: 1.5109 - acc: 0.6800 - val_loss: 1.4620 - val_acc: 0.6400
Epoch 3/200
100/100 [==============================] - 0s 212us/sample - loss: 1.2076 - acc: 0.6800 - val_loss: 1.1421 - val_acc: 0.6400
Epoch 4/200
100/100 [==============================] - 0s 173us/sample - loss: 0.9735 - acc: 0.6800 - val_loss: 0.9348 - val_acc: 0.3600
Epoch 5/200
100/100 [==============================] - 0s 186us/sample - loss: 0.8608 - acc: 0.4900 - val_loss: 0.9240 - val_acc: 0.5800
Epoch 6/200
100/100 [==============================] - 0s 180us/sample - loss: 0.8404 - acc: 0.5300 - val_loss: 0.9539 - val_acc: 0.6400
Epoch 7/200
100/100 [==============================] - 0s 280us/sample - loss: 0.8494 - acc: 0.6600 - val_loss: 0.8714 - val_acc: 0.400

### Model Prediction

In [296]:
model.predict(X_test)

array([[0.0070964 , 0.42534918, 0.5675545 ],
       [0.02109862, 0.4930267 , 0.48587465],
       [0.95229745, 0.04307787, 0.0046247 ],
       [0.02403436, 0.5062848 , 0.46968088],
       [0.00719629, 0.4027993 , 0.5900044 ],
       [0.83974093, 0.13664341, 0.02361562],
       [0.03065607, 0.55730265, 0.41204122],
       [0.0379925 , 0.54041326, 0.42159417],
       [0.91847664, 0.07226035, 0.00926301],
       [0.04800338, 0.5632862 , 0.38871047],
       [0.01636266, 0.5056939 , 0.47794342],
       [0.04029114, 0.53937674, 0.42033213],
       [0.9710118 , 0.02672295, 0.00226521],
       [0.00215683, 0.35363838, 0.6442048 ],
       [0.9506299 , 0.04475385, 0.00461625],
       [0.01472945, 0.47059742, 0.5146731 ],
       [0.00210521, 0.3721101 , 0.62578464],
       [0.00539348, 0.42247972, 0.5721268 ],
       [0.94971335, 0.0455059 , 0.00478075],
       [0.92213464, 0.06941833, 0.00844705],
       [0.0780825 , 0.5756609 , 0.3462566 ],
       [0.0031218 , 0.37730134, 0.6195768 ],
       [0.

### Save the Model

In [0]:
model.save('iris_final.h5')

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?